In [25]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Understand Plans and DAG")
    .master("local[*]")
    .getOrCreate()
)

spark

In [26]:
# Disable AQE and Broadcast join

# AQE is a feature that optimizes query plans dynamically based on runtime statistics.
# By setting this to False, Spark won’t change partition sizes or join strategies at runtime.
spark.conf.set("spark.sql.adaptive.enabled", False)

# AQE can merge small partitions into larger ones for efficiency.
# Disabling this ensures partitioning remains as originally planned, avoiding surprises if your downstream logic depends on specific partitioning.
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)

# Normally, Spark automatically broadcasts small tables to all executors for faster joins.
# Setting this to -1 disables automatic broadcast joins, so Spark will use shuffle joins instead.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# Broadcast join: sends small table to all nodes -> no shuffle, very fast (use when one table is small)
# Shuffle join: redistributes both tables by key -> involves network shuffle, slower but works for large tables
# Spark auto-selects, or force broadcast using: df_large.join(broadcast(df_small), "key")

In [27]:
# Check default Parallism

spark.sparkContext.defaultParallelism

8

In [28]:
# Create dataframes

df_1 = spark.range(4, 200, 2)
df_2 = spark.range(2, 200, 4)

In [29]:
df_1.rdd.getNumPartitions()
#df_2.rdd.getNumPartitions()

8

In [30]:
# Re-partition data

df_3 = df_1.repartition(5)
df_4 = df_2.repartition(7)

In [31]:
df_3.rdd.getNumPartitions()

5

In [32]:
df_4.rdd.getNumPartitions()

7

In [43]:
# Join the dataframes

df_joined = df_3.join(df_4, on="id")

# Default number of shuffle partitions is 200

# Check current value
print(spark.conf.get("spark.sql.shuffle.partitions"))  # Default: 200

# Change it (e.g., to 50 for smaller datasets)
spark.conf.set("spark.sql.shuffle.partitions", 50)
print(spark.conf.get("spark.sql.shuffle.partitions"))

200
50


In [44]:
# Get the sum of ids
#df_sum = df_joined.selectExpr("sum(id) as total_sum")

from pyspark.sql.functions import sum

df_sum = df_joined.select(sum("id").alias("total_sum"))

In [45]:
# View data

df_sum.show()

+---------+
|total_sum|
+---------+
|     4998|
+---------+



In [36]:
# Explain plan

df_sum.explain()

== Physical Plan ==
*(6) HashAggregate(keys=[], functions=[sum(id#37L)])
+- Exchange SinglePartition, ENSURE_REQUIREMENTS, [id=#392]
   +- *(5) HashAggregate(keys=[], functions=[partial_sum(id#37L)])
      +- *(5) Project [id#37L]
         +- *(5) SortMergeJoin [id#37L], [id#39L], Inner
            :- *(2) Sort [id#37L ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(id#37L, 200), ENSURE_REQUIREMENTS, [id=#376]
            :     +- Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [id=#375]
            :        +- *(1) Range (4, 200, step=2, splits=8)
            +- *(4) Sort [id#39L ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(id#39L, 200), ENSURE_REQUIREMENTS, [id=#383]
                  +- Exchange RoundRobinPartitioning(7), REPARTITION_BY_NUM, [id=#382]
                     +- *(3) Range (2, 200, step=4, splits=8)




In [46]:
# Union the data again to see the skipped stages

df_union = df_sum.union(df_4)

In [47]:
df_union.show()

+---------+
|total_sum|
+---------+
|     4998|
|       14|
|       38|
|       50|
|       74|
|      110|
|      130|
|      154|
|      186|
|       10|
|       30|
|       54|
|       98|
|      118|
|      138|
|      158|
|      178|
|        6|
|       42|
|       70|
+---------+
only showing top 20 rows



In [48]:
# Explain plan

df_union.explain()

== Physical Plan ==
Union
:- *(6) HashAggregate(keys=[], functions=[sum(id#37L)])
:  +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [id=#721]
:     +- *(5) HashAggregate(keys=[], functions=[partial_sum(id#37L)])
:        +- *(5) Project [id#37L]
:           +- *(5) SortMergeJoin [id#37L], [id#39L], Inner
:              :- *(2) Sort [id#37L ASC NULLS FIRST], false, 0
:              :  +- Exchange hashpartitioning(id#37L, 50), ENSURE_REQUIREMENTS, [id=#705]
:              :     +- Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [id=#704]
:              :        +- *(1) Range (4, 200, step=2, splits=8)
:              +- *(4) Sort [id#39L ASC NULLS FIRST], false, 0
:                 +- Exchange hashpartitioning(id#39L, 50), ENSURE_REQUIREMENTS, [id=#712]
:                    +- Exchange RoundRobinPartitioning(7), REPARTITION_BY_NUM, [id=#711]
:                       +- *(3) Range (2, 200, step=4, splits=8)
+- ReusedExchange [id#76L], Exchange RoundRobinPartitioning(7), REPARTITI

In [50]:
# Above transformation will have skipped stages as all the stages in df_sum was already processed
# DAG Optimizations (Catalyst)
# Spark’s query planner may:
# Combine stages (pipelining narrow transformations)
# Eliminate redundant computations
# Push filters or projections down

In [ ]:
# In Spark, shuffle (exchange) writes and reads data across nodes:
# 1. Shuffle Write: tasks write output partitioned by key to disk/memory
# 2. Shuffle Read: downstream tasks read needed partitions for processing
# Triggered by wide transformations like join, groupBy, reduceByKey, distinct

# Benefits of shuffle/exchange in Spark:
# 1. Enables wide transformations like join, groupBy, distinct across partitions
# 2. Redistributes data so related keys end up on the same node
# 3. Allows parallel processing of large datasets across the cluster
# 4. Supports fault tolerance by storing intermediate shuffle data

In [49]:
# DataFrame to RDD

df_1.rdd

MapPartitionsRDD[5] at javaToPython at NativeMethodAccessorImpl.java:0